### MLflow

In [ ]:
import mlflow
import mlflow.onnx
import onnxruntime as rt
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import onnx

In [ ]:
# Load the ONNX model
onnx_model_path = "htmlphishcnnattention.onnx"
onnx_model = onnx.load(onnx_model_path)

# Set up ONNX Runtime for inference
sess = rt.InferenceSession(onnx_model_path)

# Check the input metadata to verify the expected shape and type
input_name = sess.get_inputs()[0].name  # 'input'
input_shape = sess.get_inputs()[0].shape  # This will now be dynamic
input_type = sess.get_inputs()[0].type
print(f"Input Name: {input_name}")
print(f"Expected Input Shape: {input_shape} (Dynamic)")
print(f"Expected Input Type: {input_type}")

# Sample test data (replace with your actual test data)
X_test = test_char_padded_windows  # Test data (e.g., [batch_size, sequence_length])
y_test = test_label_windows_tensor  # Replace with actual true labels

# Ensure X_test is a numpy array and has the correct type
if not isinstance(X_test, np.ndarray):
    X_test = np.array(X_test)

X_test = X_test.astype(np.int64)  # Use int64, as the input is most likely integer token indices (for sequence data)

# Send the entire test tensor at once
print(f"Sending entire batch of shape: {X_test.shape}")
predictions = sess.run(None, {input_name: X_test})[0]

# Binarize predictions if necessary
y_pred = np.argmax(predictions, axis=1)  # Adjust based on your model's output format

# Calculate metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average="weighted")
recall = recall_score(y_test, y_pred, average="weighted")
f1 = f1_score(y_test, y_pred, average="weighted")
conf_matrix = confusion_matrix(y_test, y_pred)

# Log metrics and confusion matrix in MLflow
mlflow.set_experiment("Capstone ONNX Model Metrics")

with mlflow.start_run():
    # Log the ONNX model
    mlflow.onnx.log_model(onnx_model, "onnx_model")

    # Log metrics
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1_score", f1)

    # Plot confusion matrix
    plt.figure(figsize=(8, 6))
    sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues")
    plt.title("Confusion Matrix")
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.savefig("confusion_matrix.png")

    # Log confusion matrix as an artifact
    mlflow.log_artifact("confusion_matrix.png")

In [ ]:
import subprocess
# Start MLflow UI in the background
subprocess.Popen(["mlflow", "ui", "--port", "5001"])